<a href="https://colab.research.google.com/github/FaizaIrfan0/C_lab_programs/blob/main/Lang_models(bigram_and_transformer_arcitecture).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BIGRAM

In [ ]:
corpus=['i am happy',
        'i am sad',
        'you are happy',
        'you are sad']
Corpus=[]
for i in corpus:
  i="<s> "+i+" <e>"
  Corpus.append(i)
print(Corpus)
bigram=[]
i=0
j=0
for i in Corpus:
  words=i.split()
  for j in range(len(words)-1):
    bigram.append((words[j],words[j+1]))
print(bigram)

count_dict = {}
for bg in bigram:
  if bg in count_dict:
    count_dict[bg] += 1
  else:
    count_dict[bg] = 1
print("counts:", count_dict)


from collections import defaultdict
counts = defaultdict(lambda: defaultdict(int))
for (w1, w2), count in count_dict.items():
    counts[w1][w2] = count

probs = {}

for w1 in counts:
    total = sum(counts[w1].values())
    probs[w1] = {}

    for w2 in counts[w1]:
      probs[w1][w2] = counts[w1][w2] / total
print(probs)

def implementing_bigram_model():
  word=input("enter word:")
  if word not in probs:
    print("enter valid word")
  else:
    import random
    next_words = list(probs[word].keys())
    probabilities = list(probs[word].values())
    print(random.choices(next_words, probabilities)[0])

implementing_bigram_model()

['<s> i am happy <e>', '<s> i am sad <e>', '<s> you are happy <e>', '<s> you are sad <e>']
[('<s>', 'i'), ('i', 'am'), ('am', 'happy'), ('happy', '<e>'), ('<s>', 'i'), ('i', 'am'), ('am', 'sad'), ('sad', '<e>'), ('<s>', 'you'), ('you', 'are'), ('are', 'happy'), ('happy', '<e>'), ('<s>', 'you'), ('you', 'are'), ('are', 'sad'), ('sad', '<e>')]
counts: {('<s>', 'i'): 2, ('i', 'am'): 2, ('am', 'happy'): 1, ('happy', '<e>'): 2, ('am', 'sad'): 1, ('sad', '<e>'): 2, ('<s>', 'you'): 2, ('you', 'are'): 2, ('are', 'happy'): 1, ('are', 'sad'): 1}
{'<s>': {'i': 0.5, 'you': 0.5}, 'i': {'am': 1.0}, 'am': {'happy': 0.5, 'sad': 0.5}, 'happy': {'<e>': 1.0}, 'sad': {'<e>': 1.0}, 'you': {'are': 1.0}, 'are': {'happy': 0.5, 'sad': 0.5}}
enter word:you
are


In [ ]:
import nltk
nltk.download('brown')

from nltk.corpus import brown

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [ ]:
sentences=brown.sents()
for i in range(10):
  print(sentences[i])

['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']
['The', 'jury', 'further', 'said', 'in', 'term-end', 'presentments', 'that', 'the', 'City', 'Executive', 'Committee', ',', 'which', 'had', 'over-all', 'charge', 'of', 'the', 'election', ',', '``', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'City', 'of', 'Atlanta', "''", 'for', 'the', 'manner', 'in', 'which', 'the', 'election', 'was', 'conducted', '.']
['The', 'September-October', 'term', 'jury', 'had', 'been', 'charged', 'by', 'Fulton', 'Superior', 'Court', 'Judge', 'Durwood', 'Pye', 'to', 'investigate', 'reports', 'of', 'possible', '``', 'irregularities', "''", 'in', 'the', 'hard-fought', 'primary', 'which', 'was', 'won', 'by', 'Mayor-nominate', 'Ivan', 'Allen', 'Jr.', '.']
['``', 'Only', 'a', 'relative', 'handful', 'of', 'such', 'reports

TRANSFORMERS

1. SCALED DOT PRODUCT ATTENTION

input an embedding vector(e) --> randomly initialise three matices, Wq, Wk and Wv. --> multiply embedding vector with each of the matrices to get Q, K and V. --> compute QK.T --> apply softmax to the output obtained --> and multiply with V.

In [ ]:
"""import numpy as np
import random as rd"""

In [ ]:
"""e=np.array([
    [0.5,0.1,0.2,0.5,0.7,0.9,0.1,0.3],
     [0.1,0.3,0.2,0.5,0.8,0.5,0.4,0.7],
      [0.6,0.7,0.2,0.9,0.5,0.4,0.8,0.1]])
dim=e.shape[1]
Wq=np.random.randn(dim,dim)
Wk=np.random.randn(dim,dim)
Wv=np.random.randn(dim,dim)
Eq=np.matmul(e,Wq)
Ek=np.matmul(e,Wk)
Ev=np.matmul(e,Wv)
print(f"Eq= {Eq} \n Ek= {Ek} \n Ev= {Ev}")"""

Eq= [[-2.05764566 -1.77591594 -0.95367368 -1.28764161  0.93902232 -0.33780294
   1.76326019  0.65014164]
 [-2.06847131 -2.6247433  -1.22304471 -1.21556697 -0.53792224 -0.75614303
   1.05174792 -0.15151298]
 [-2.43311308 -1.70099953 -1.54253696 -1.39314117 -0.79173779 -2.35970085
  -0.67522238 -0.75521208]] 
 Ek= [[-2.11060278 -0.04671447  1.22222838  1.19453426 -4.29964737 -0.75944163
  -2.2373985  -0.28066591]
 [-2.08724892  0.84231447  0.60754495  0.6395289  -2.18397706  0.07792438
  -0.78428594 -0.9738213 ]
 [-3.14745926  0.69365027 -0.82990897 -0.01699406 -2.70755303  1.19656947
  -1.61792124  0.39073275]] 
 Ev= [[-2.9204475  -0.147638   -0.78227449 -2.81819249  0.49302813 -0.52664702
  -0.06987438  0.2451554 ]
 [-3.60780332  0.24991203 -0.59137235 -2.23572136  0.99292396 -1.37852163
  -1.35120217  0.80220978]
 [-4.6970309  -1.60731919 -0.20220859 -3.50805214  2.72296412 -0.21751039
   0.72781949  0.82002257]]


In [ ]:
"""PreMax=np.matmul(Eq,Ek.T)
print(PreMax)"""

[[-6.18641698 -2.69709293  0.51239402]
 [ 2.11792623  1.02467373  4.51627745]
 [ 8.5842614   4.62788951  7.89956248]]


In [ ]:
"""from math import sqrt
PreMax/=sqrt(dim)
print(PreMax)"""

[[-2.1872287  -0.95356635  0.18115864]
 [ 0.7488      0.36227687  1.59674521]
 [ 3.03499472  1.63620603  2.7929171 ]]


In [ ]:
"""from scipy.special import softmax
SoftMax=softmax(PreMax, axis=1)
print(SoftMax)"""

[[0.0661641  0.2271931  0.7066428 ]
 [0.24911196 0.1692504  0.58163764]
 [0.49215236 0.12151038 0.38633726]]


In [ ]:
"""result=np.matmul(SoftMax,Ev)
print(result)"""

[[-4.33201988 -1.08879058 -0.32900345 -3.17334343  2.18236923 -0.50173788
   0.20270142  0.77794006]
 [-4.07011052 -0.92935801 -0.41257607 -3.12085735  1.8746504  -0.49102164
   0.17722915  0.67380146]
 [-3.69032873 -0.66326078 -0.53497683 -3.0139347   1.41527803 -0.51072762
   0.08260987  0.5349359 ]]


MULTI HEAD ATTENTION IN NUMPY

The goal for n head multi attention is to generate n output vectors to capture n distinct semantic meanings of the input content.

In [ ]:
"""n=4
din=int(dim/n)
wq=[]
wv=[]
wk=[]
for i in range(n):
  wq.append(np.random.randn(dim,din))
  wv.append(np.random.randn(dim,din))
  wk.append(np.random.randn(dim,din))
print(wq)"""

[array([[ 1.1441743 , -0.82603443],
       [ 0.28827043, -1.15592013],
       [ 1.43848359,  0.05122885],
       [ 0.41719621, -1.82289343],
       [ 1.92574573, -0.74556532],
       [-0.16965822,  1.52399567],
       [-1.08965439,  1.18698195],
       [-0.51703345,  1.05266323]]), array([[-0.38546   , -0.03481308],
       [ 0.60591273,  0.88210602],
       [-0.56683238,  0.45603442],
       [ 0.6702429 ,  0.05188497],
       [-1.06678217, -0.16971277],
       [-0.70422845, -0.6804739 ],
       [-0.03040093, -0.98979493],
       [-0.55827578,  0.47983336]]), array([[ 0.47903223,  0.28494121],
       [-0.92358155, -1.74908499],
       [-1.28994618, -0.29851117],
       [ 0.60890116, -0.92180561],
       [-1.08910166,  1.21784019],
       [-1.0515122 ,  0.96438308],
       [-1.42582552,  0.11162735],
       [ 0.15735855, -0.81774641]]), array([[-0.85829691,  0.08476106],
       [ 0.97068172,  0.48915418],
       [-1.44256918, -0.44574264],
       [-0.90591733, -1.3014813 ],
       [-0.94

In [ ]:
"""eq=[]
ek=[]
ev=[]
for i in range(n):
  eq.append(np.matmul(e,wq[i]))
  ek.append(np.matmul(e,wk[i]))
  ev.append(np.matmul(e,wv[i]))"""


In [ ]:
"""Epre=[]
for i in range(n):
  Epre.append((np.matmul(eq[i],ek[i].T))/sqrt(dim))
print(Epre)"""

[array([[-1.47557017, -1.69592926, -1.96873594],
       [-1.0047134 , -1.14375846, -1.27053869],
       [-0.90627253, -1.15885168, -1.95510983]]), array([[-0.69030581, -0.4598424 , -0.36514796],
       [-0.64681261, -0.46820607, -0.29366193],
       [-0.034741  ,  0.01807857, -0.0719005 ]]), array([[-0.07375244,  0.16081046, -0.52982686],
       [ 0.45892382,  0.55772966, -0.54212199],
       [ 0.85415249,  0.82646605, -0.48777412]]), array([[ 0.25438252,  0.23766158,  0.57716847],
       [-0.12806671,  0.03935685, -0.36541203],
       [-0.43811616, -0.1334116 , -1.12390671]])]


In [ ]:
"""Epost=[]
Epost=softmax(Epre, axis=1)

result=np.matmul(Epost,ev)
print(result)
final_result=np.concatenate(result,axis=1)
print(f"\n \nfinal output is : {final_result}")"""

[[[ 0.39108466 -0.1295758 ]
  [ 0.62123094 -0.19527497]
  [ 0.74451521 -0.26247752]]

 [[ 0.92824293  2.17407478]
  [ 0.95597857  2.24757829]
  [ 1.60336784  3.43422091]]

 [[ 0.62086998  1.11945064]
  [ 0.89470904  1.356296  ]
  [ 1.20817963  1.64297704]]

 [[-1.6880475   1.07881828]
  [-1.03927921  0.78666235]
  [-0.74263865  0.61332367]]]

 
final output is : [[ 0.39108466 -0.1295758   0.92824293  2.17407478  0.62086998  1.11945064
  -1.6880475   1.07881828]
 [ 0.62123094 -0.19527497  0.95597857  2.24757829  0.89470904  1.356296
  -1.03927921  0.78666235]
 [ 0.74451521 -0.26247752  1.60336784  3.43422091  1.20817963  1.64297704
  -0.74263865  0.61332367]]


In [ ]:
"""def multi_head_attention(e,n):
  n_samples,n_dims=e.shape
  din=int(n_dims/n)
  wq=[]
  wv=[]
  wk=[]
  for i in range(n):
    wq.append(np.random.randn(dim,din))
    wv.append(np.random.randn(dim,din))
    wk.append(np.random.randn(dim,din))
  eq=[]
  ek=[]
  ev=[]
  for i in range(n):
    eq.append(np.matmul(e,wq[i]))
    ek.append(np.matmul(e,wk[i]))
    ev.append(np.matmul(e,wv[i]))
  epre=[]
  for i in range(n):
   epre.append((np.matmul(eq[i],ek[i].T))/sqrt(dim))
  Epost=[]
  Epost=softmax(Epre, axis=1)

  result=np.matmul(Epost,ev)
  final_result=np.concatenate(result,axis=1)
  return final_result
  return e"""

TRANSFORMER ENCODER IN NUMPY


---

input sentences

tokenisation

embedding

positional encoding

multi head attention

addition and normalisation

mlp

addition and normalisation





In [ ]:
"""n_seq,n_dim=e.shape
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / (10000 ** (2 * (i // 2) / d_model))
    angles = pos * angle_rates

    PE = np.zeros((seq_len, d_model))
    PE[:, 0::2] = np.sin(angles[:, 0::2])
    PE[:, 1::2] = np.cos(angles[:, 1::2])

    return PE
PosEn=positional_encoding(n_seq,n_dim)
if PosEn.shape==e.shape:
  e+=PosEn
n=4"""

In [ ]:
"""result_1=multi_head_attention(e,n)"""

In [ ]:
"""add_1=result_1+e
import torch.nn as nn
add_1_tensor = torch.tensor(add_1, dtype=torch.float32)
ln=nn.LayerNorm(8)
norm_1=ln(add_1_tensor)"""

In [ ]:
"""#feed forward neural network
d_model = e.shape[1]
d_ff = 4*d_model

ffn = nn.Sequential(
    nn.Linear(d_model, d_ff),
    nn.ReLU(),
    nn.Linear(d_ff, d_model)
            )
out_ffn=ffn(norm_1)
print(out_ffn)"""

tensor([[-0.1792,  0.2349, -0.1589, -0.0538,  0.2972,  0.2110, -0.2866, -0.3955],
        [-0.2929,  0.1679, -0.1586, -0.1122,  0.3413,  0.0863, -0.0137, -0.4521],
        [-0.3338,  0.1550, -0.2403, -0.1408,  0.4057,  0.1070, -0.0714, -0.4772]],
       grad_fn=<AddmmBackward0>)


In [ ]:
"""add_2=norm_1+out_ffn
print(add_2)"""

tensor([[ 1.5040, -0.5118, -0.3760,  1.4233,  0.6784, -0.6053, -1.0740, -1.3694],
        [ 1.6677, -1.2426, -0.3665,  0.8733,  0.4833, -0.9243, -0.2950, -0.6300],
        [ 1.4249, -1.0198, -0.5939,  1.1518,  0.3820, -1.2003, -0.1287, -0.6117]],
       grad_fn=<AddBackward0>)


In [ ]:
"""norm_2=ln(add_2)
print(norm_2,"\n",norm_2.shape)"""

tensor([[ 1.4963, -0.4555, -0.3240,  1.4182,  0.6969, -0.5461, -0.9999, -1.2859],
        [ 1.8718, -1.2917, -0.3394,  1.0083,  0.5843, -0.9457, -0.2616, -0.6258],
        [ 1.6404, -1.0344, -0.5683,  1.3417,  0.4994, -1.2318, -0.0593, -0.5878]],
       grad_fn=<NativeLayerNormBackward0>) 
 torch.Size([3, 8])


END TO END TRANSFORMER ARCITECTURE IN PYTORCH

TRANSFORMER ENCODER

In [ ]:
import torch
import torch.nn as nn

def create_causal_mask(seq_len):
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask


class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads):
    super().__init__()
    assert d_model % num_heads == 0
    self.num_heads = num_heads
    self.d_k = d_model // num_heads

    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model, d_model)

  def scaled_dot_product(self, Q, K, V, mask=None):
    scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
    if mask is not None:
      scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V

  def forward(self, Q, K, V, mask=None):
    B, S, D = Q.shape

    Q = self.W_q(Q).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
    K = self.W_k(K).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
    V = self.W_v(V).view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    out = self.scaled_dot_product(Q, K, V, mask)
    out = out.transpose(1, 2).contiguous().view(B, S, D)
    return self.W_o(out)


class EncoderBlock(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()
    self.attention = MultiHeadAttention(d_model, num_heads)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.ffn = nn.Sequential(
        nn.Linear(d_model, d_ff, d_model),
        nn.ReLU(),
        nn.Linear(d_ff, d_model)
        )
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, mask=None):
    attn_out = self.attention(x, x, x, mask)
    x = self.norm1(x + self.dropout(attn_out))

                                                                                                                                                                       # FFN + Add & Norm
    ffn_out = self.ffn(x)
    x = self.norm2(x + self.dropout(ffn_out))
    return x

class TransformerEncoder(nn.Module):
  def __init__(self, vocab_size, d_model=8, num_heads=4, d_ff=32, dropout=0.1):
    super().__init__()
    self.block = EncoderBlock(d_model, num_heads, d_ff, dropout)
    self.embedding = nn.Embedding(vocab_size, d_model)

  def positional_encoding(self, seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1)
    i = torch.arange(d_model).unsqueeze(0)
    angle_rates = 1 / (10000 ** (2 * (i // 2) / d_model))
    angles = pos * angle_rates
    PE = torch.zeros(seq_len, d_model)
    PE[:, 0::2] = torch.sin(angles[:, 0::2])
    PE[:, 1::2] = torch.cos(angles[:, 1::2])
    return PE

  def forward(self, x):
    x=self.embedding(x)
    PE = self.positional_encoding(x.shape[1], x.shape[2])
    x = x + PE
    return self.block(x)

if __name__ == "__main__":
  encoder = TransformerEncoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
  x = torch.randint(0, 100, (1, 3))  # random token IDs
  out = encoder(x)
  print(out.shape)  # should be torch.Size([1, 3, 8])


torch.Size([1, 3, 8])


TRANSFORMER DECODER

In [ ]:
class DecoderBlock(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()
    self.attention1=MultiHeadAttention(d_model,num_heads)
    self.attention2=MultiHeadAttention(d_model,num_heads)
    self.norm1=nn.LayerNorm(d_model)
    self.norm2=nn.LayerNorm(d_model)
    self.norm3=nn.LayerNorm(d_model)
    self.ffn=nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.ReLU(),
        nn.Linear(d_ff, d_model)
    )
    self.dropout=nn.Dropout(dropout)
  def forward(self, x, enc_output, mask=None):

    attn_out1=self.attention1(x,x,x,mask)
    x=self.norm1(x+self.dropout(attn_out1))

    attn_out2=self.attention2(x, enc_output, enc_output, None)
    x=self.norm2(x+self.dropout(attn_out2))

    ffn_out=self.ffn(x)
    x=self.norm3(x+self.dropout(ffn_out))

    return x

class TransformerDecoder(nn.Module):
  def __init__(self, vocab_size, d_model=8, num_heads=4, d_ff=32, dropout=0.1):
    super().__init__()
    self.block = DecoderBlock(d_model, num_heads, d_ff, dropout)
    self.embedding = nn.Embedding(vocab_size, d_model)

  def positional_encoding(self, seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1)
    i = torch.arange(d_model).unsqueeze(0)
    angle_rates = 1 / (10000 ** (2 * (i // 2) / d_model))
    angles = pos * angle_rates
    PE = torch.zeros(seq_len, d_model)
    PE[:, 0::2] = torch.sin(angles[:, 0::2])
    PE[:, 1::2] = torch.cos(angles[:, 1::2])
    return PE

  def forward(self, x, enc_output, mask=None):
    x=self.embedding(x)
    PE = self.positional_encoding(x.shape[1], x.shape[2])
    x = x + PE
    return self.block(x, enc_output, mask)

if __name__ == "__main__":
  decoder = TransformerDecoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
  x=torch.randint(0, 100, (1,3))
  enc_output = torch.randn(1,3,8)  # random token IDs
  mask=create_causal_mask(3)
  out = decoder(x,enc_output, mask)
  print(out.shape)  # should be torch.Size([1, 3, 8])


torch.Size([1, 3, 8])


ENCODER+DECODER

In [ ]:
class Transformer(nn.Module):
  def __init__(self, encoder, decoder, vocab_size, d_model):
    super().__init__()
    self.encoder=encoder
    self.decoder=decoder
    self.projection = nn.Linear(d_model, vocab_size)
  def forward(self, src, tgt, mask=None):
    enc_output=self.encoder(src)
    out=self.decoder(tgt, enc_output)
    return self.projection(out)

if __name__ == "__main__":
    encoder = TransformerEncoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
    decoder = TransformerDecoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
    model = Transformer(encoder, decoder,vocab_size=100, d_model=8)
    src = torch.randint(0, 100, (1, 3))
    tgt = torch.randint(0, 100, (1, 3))
    mask=create_causal_mask(3)
    out = model(src, tgt, mask)
    print(out.shape)



torch.Size([1, 3, 100])


In [ ]:
encoder = TransformerEncoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
decoder = TransformerDecoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
model = Transformer(encoder, decoder, vocab_size=100, d_model=8)
src = torch.randint(0, 100, (1, 3))
tgt = torch.randint(0, 100, (1, 3))
mask = create_causal_mask(3)
out = model(src, tgt, mask)
print(out.shape)

torch.Size([1, 3, 100])


In [ ]:
import torch
import torch.nn as nn

#encoder

def create_causal_mask(seq_len):
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask


class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads):
    super().__init__()
    assert d_model % num_heads == 0
    self.num_heads = num_heads
    self.d_k = d_model // num_heads

    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model, d_model)

  def scaled_dot_product(self, Q, K, V, mask=None):
    scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
    if mask is not None:
      scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V

  def forward(self, Q, K, V, mask=None):
    B, S, D = Q.shape

    Q = self.W_q(Q).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
    K = self.W_k(K).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
    V = self.W_v(V).view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    out = self.scaled_dot_product(Q, K, V, mask)
    out = out.transpose(1, 2).contiguous().view(B, S, D)
    return self.W_o(out)


class EncoderBlock(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()
    self.attention = MultiHeadAttention(d_model, num_heads)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.ffn = nn.Sequential(
        nn.Linear(d_model, d_ff, d_model),
        nn.ReLU(),
        nn.Linear(d_ff, d_model)
        )
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, mask=None):
    attn_out = self.attention(x, x, x, mask)
    x = self.norm1(x + self.dropout(attn_out))

                                                                                                                                                                       # FFN + Add & Norm
    ffn_out = self.ffn(x)
    x = self.norm2(x + self.dropout(ffn_out))
    return x

class TransformerEncoder(nn.Module):
  def __init__(self, vocab_size, d_model=8, num_heads=4, d_ff=32, dropout=0.1):
    super().__init__()
    self.block = EncoderBlock(d_model, num_heads, d_ff, dropout)
    self.embedding = nn.Embedding(vocab_size, d_model)

  def positional_encoding(self, seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1)
    i = torch.arange(d_model).unsqueeze(0)
    angle_rates = 1 / (10000 ** (2 * (i // 2) / d_model))
    angles = pos * angle_rates
    PE = torch.zeros(seq_len, d_model)
    PE[:, 0::2] = torch.sin(angles[:, 0::2])
    PE[:, 1::2] = torch.cos(angles[:, 1::2])
    return PE

  def forward(self, x):
    x=self.embedding(x)
    PE = self.positional_encoding(x.shape[1], x.shape[2])
    x = x + PE
    return self.block(x)

if __name__ == "__main__":
  encoder = TransformerEncoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
  x = torch.randint(0, 100, (1, 3))  # random token IDs
  out = encoder(x)
  print(out.shape)  # should be torch.Size([1, 3, 8])

#decoder


class DecoderBlock(nn.Module):
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()
    self.attention1=MultiHeadAttention(d_model,num_heads)
    self.attention2=MultiHeadAttention(d_model,num_heads)
    self.norm1=nn.LayerNorm(d_model)
    self.norm2=nn.LayerNorm(d_model)
    self.norm3=nn.LayerNorm(d_model)
    self.ffn=nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.ReLU(),
        nn.Linear(d_ff, d_model)
    )
    self.dropout=nn.Dropout(dropout)
  def forward(self, x, enc_output, mask=None):

    attn_out1=self.attention1(x,x,x,mask)
    x=self.norm1(x+self.dropout(attn_out1))

    attn_out2=self.attention2(x, enc_output, enc_output, None)
    x=self.norm2(x+self.dropout(attn_out2))

    ffn_out=self.ffn(x)
    x=self.norm3(x+self.dropout(ffn_out))

    return x


class TransformerDecoder(nn.Module):
  def __init__(self, vocab_size, d_model=8, num_heads=4, d_ff=32, dropout=0.1):
    super().__init__()
    self.block = DecoderBlock(d_model, num_heads, d_ff, dropout)
    self.embedding = nn.Embedding(vocab_size, d_model)

  def positional_encoding(self, seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1)
    i = torch.arange(d_model).unsqueeze(0)
    angle_rates = 1 / (10000 ** (2 * (i // 2) / d_model))
    angles = pos * angle_rates
    PE = torch.zeros(seq_len, d_model)
    PE[:, 0::2] = torch.sin(angles[:, 0::2])
    PE[:, 1::2] = torch.cos(angles[:, 1::2])
    return PE

  def forward(self, x, enc_output, mask=None):
    x=self.embedding(x)
    PE = self.positional_encoding(x.shape[1], x.shape[2])
    x = x + PE
    return self.block(x, enc_output, mask)

if __name__ == "__main__":
  decoder = TransformerDecoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
  x=torch.randint(0, 100, (1,3))
  enc_output = torch.randn(1,3,8)  # random token IDs
  mask=create_causal_mask(3)
  out = decoder(x,enc_output, mask)
  print(out.shape)  # should be torch.Size([1, 3, 8])

#encoder+decoder

class Transformer(nn.Module):
  def __init__(self, encoder, decoder, vocab_size, d_model):
    super().__init__()
    self.encoder=encoder
    self.decoder=decoder
    self.projection = nn.Linear(d_model, vocab_size)
  def forward(self, src, tgt, mask=None):
    enc_output=self.encoder(src)
    out=self.decoder(tgt, enc_output)
    return self.projection(out)

if __name__ == "__main__":
    encoder = TransformerEncoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
    decoder = TransformerDecoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
    model = Transformer(encoder, decoder,vocab_size=100, d_model=8)
    src = torch.randint(0, 100, (1, 3))
    tgt = torch.randint(0, 100, (1, 3))
    mask=create_causal_mask(3)
    out = model(src, tgt, mask)
    print(out.shape)

#test

encoder = TransformerEncoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
decoder = TransformerDecoder(vocab_size=100, d_model=8, num_heads=4, d_ff=32)
model = Transformer(encoder, decoder, vocab_size=100, d_model=8)
src = torch.randint(0, 100, (1, 3))
tgt = torch.randint(0, 100, (1, 3))
mask = create_causal_mask(3)
out = model(src, tgt, mask)
print(out.shape)

torch.Size([1, 3, 8])
torch.Size([1, 3, 8])
torch.Size([1, 3, 100])
torch.Size([1, 3, 100])
